# Susie benchamrking

In [3]:
import numpy as np
import pandas as pd
import pyspark.sql.functions as f
from pyspark.sql import DataFrame, Window

from gentropy.common.session import Session
from gentropy.dataset.study_index import StudyIndex
from gentropy.dataset.study_locus import StudyLocus
from gentropy.dataset.summary_statistics import SummaryStatistics
from gentropy.datasource.gnomad.ld import GnomADLDMatrix
from gentropy.method.susie_inf import SUSIE_inf

In [165]:
ld = np.loadtxt("../tests/gentropy/data_samples/01_test_ld.csv", delimiter=",")
z = pd.read_csv("../tests/gentropy/data_samples/01_test_z.csv")
z = np.array(z.iloc[:, 1])
#z[5]=-z[5]

In [148]:
session = Session()
gwas2=SummaryStatistics.from_parquet(session, "../tests/gentropy/data_samples/sumstats_sample/GCST005523_chr18.parquet")
gwas_df=gwas2._df.limit(21)

In [166]:
susie_output = SUSIE_inf.susie_inf(z=z, LD=ld, L=10,est_tausq=False)

In [153]:
susie_output.keys()
sum(susie_output["PIP"][:,0])

0.9999999999999982

In [154]:
susie_output["lbf"]

array([22.45514326,  3.39215789, -0.03210043, -0.03172559, -0.03150664,
       -0.03142112, -0.0314487 , -0.03157488, -0.03179368, -0.03210934])

In [22]:
_join=gwas_df
_variants = np.array(
    [row["variantId"] for row in _join.select("variantId").collect()]
)

In [25]:
variants = _variants.reshape(-1, 1)
PIPs = susie_output["PIP"]
lbfs = susie_output["lbf_variable"]
susie_result = np.hstack((variants, PIPs, lbfs))

In [29]:
order_creds = list(enumerate(susie_output["lbf"]))
order_creds.sort(key=lambda x: x[1], reverse=True)
cred_sets = None
counter = 0

In [32]:
print(order_creds)
cs_lbf_thr=2

[(0, 731.1532533381637), (1, 351.5114266058942), (3, 126.85438855538354), (2, 51.699811264225424), (4, 37.93192592827882), (5, 31.009771681812502), (6, 23.493131948721746), (8, 16.921932293521717), (9, 9.320379511652039), (7, 1.53384939408551)]


In [33]:
i=0
value=731.1532533381637

In [70]:
sorted_arr = susie_result[
    susie_result[:, i + 1].astype(float).argsort()[::-1]
]
cumsum_arr = np.cumsum(sorted_arr[:, i + 1].astype(float))
filter_row = np.argmax(cumsum_arr >= 0.99)
if filter_row == 0 and cumsum_arr[0] < 0.99:
    filter_row = len(cumsum_arr)
filter_row += 1
filtered_arr = sorted_arr[:filter_row]

In [72]:
cred_set = filtered_arr[:, [0, i + 1, i + L_snps+1, i + 2 * L_snps+1]]

In [39]:
filter_row = np.argmax(cumsum_arr >= 0.99)

In [75]:
cred_set

array([['18_223185_G_A', '1.0', '734.1977757758872',
        '-0.8145497584755277']], dtype='<U32')

In [76]:
cred_set = filtered_arr[:, [0, i + 1, i + L_snps + 1, i + 2 * L_snps + 1]]
win = Window.rowsBetween(
    Window.unboundedPreceding, Window.unboundedFollowing
)

In [78]:
cred_set

array([['18_223185_G_A', '1.0', '734.1977757758872',
        '-0.8145497584755277']], dtype='<U32')

In [44]:
filter_row

1

In [52]:
L_snps = PIPs.shape[1]

In [56]:
filtered_arr

array([['18_223185_G_A', '1.0', '2.855708461827948e-147', '1.0',
        '2.2409592065527667e-57', '2.1537010189865385e-19',
        '0.9999999999999858', '1.7078746882627635e-12',
        '0.003403978499650294', '3.58811302399867e-10',
        '8.745721547819502e-07', '734.1977757758872',
        '17.125260333567898', '54.74433370194885', '-0.5415353161344503',
        '-2.005480634483324', '34.05429411953591', '-0.5581170043361856',
        '-1.1044385526834652', '-1.7817697544378543',
        '-1.5846290867596602']], dtype='<U32')

In [55]:
filtered_arr = sorted_arr[:filter_row]

In [79]:
cred_set = (
                session.spark.createDataFrame(
                    cred_set.tolist(), ["variantId", "posteriorProbability", "logBF","beta"]
                )
                .join(
                    _join.select(
                        "variantId",
                        "chromosome",
                        "position",
                    ),
                    "variantId",
                ))

In [83]:
(cred_set.sort(f.desc("posteriorProbability"))
                .withColumn(
                    "locus",
                    f.collect_list(
                        f.struct(
                            "variantId",
                            "posteriorProbability",
                            "logBF",
                            "beta",
                        )
                    ).over(win),
                )
                .limit(1)).show()

24/03/21 12:03:45 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
24/03/21 12:03:45 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
24/03/21 12:03:45 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
24/03/21 12:03:45 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
24/03/21 12:03:45 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
24/03/21 12:03:45 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
24/03/21 1

In [63]:
susie_output[]

dict_keys(['PIP', 'mu', 'omega', 'lbf_variable', 'ssq', 'sigmasq', 'tausq', 'alpha', 'lbf'])

In [68]:
variants = np.array(
    [row["variantId"] for row in _join.select("variantId").collect()]
).reshape(-1, 1)
PIPs = susie_output["PIP"]
lbfs = susie_output["lbf_variable"]
mu = susie_output["mu"]
susie_result = np.hstack((variants, PIPs, lbfs, mu))

L_snps = PIPs.shape[1]

In [84]:
_studyId="lol"
region="lol1"

In [113]:
_region="lol"
cred_set = filtered_arr[:, [0, i + 1, i + L_snps + 1, i + 2 * L_snps + 1]]
win = Window.rowsBetween(
    Window.unboundedPreceding, Window.unboundedFollowing
)
cred_set = (
    session.spark.createDataFrame(
        cred_set.tolist(),
        ["variantId", "posteriorProbability", "logBF", "beta"],
    )
    .join(
        _join.select(
            "variantId",
            "chromosome",
            "position",
        ),
        "variantId",
    )
    .sort(f.desc("posteriorProbability"))
    .withColumn(
        "locus",
        f.collect_list(
            f.struct(
                f.col("variantId").cast("string").alias("variantId"),
                f.col("posteriorProbability").cast("double").alias("posteriorProbability"),
                f.col("logBF").cast("double").alias("logBF"),
                f.col("beta").cast("double").alias("beta"),
            )
        ).over(win),
    )
    .limit(1)
    .withColumns(
        {
            "studyId": f.lit(_studyId),
            "region": f.lit(_region),
            "credibleSetIndex": f.lit(i + 1),
            "credibleSetlog10BF": f.lit(value * 0.4342944819),
            "finemappingMethod": f.lit("SuSiE-inf"),
        }
    )
    .withColumn(
        "studyLocusId",
        StudyLocus.assign_study_locus_id(
            f.col("studyId"), f.col("variantId")
        ),
    )
    .select(
        "studyLocusId",
        "studyId",
        "region",
        "credibleSetIndex",
        "locus",
        "variantId",
        "chromosome",
        "position",
        "finemappingMethod",
        "credibleSetlog10BF",
    )
)

In [116]:
order_creds = list(enumerate(susie_output["lbf"]))
order_creds.sort(key=lambda x: x[1], reverse=True)
cred_sets = None
counter = 0
for i, value in order_creds:
    if counter > 0 and value < cs_lbf_thr:
        continue
    counter += 1
    sorted_arr = susie_result[
        susie_result[:, i + 1].astype(float).argsort()[::-1]
    ]
    cumsum_arr = np.cumsum(sorted_arr[:, i + 1].astype(float))
    filter_row = np.argmax(cumsum_arr >= 0.99)
    if filter_row == 0 and cumsum_arr[0] < 0.99:
        filter_row = len(cumsum_arr)
    filter_row += 1
    filtered_arr = sorted_arr[:filter_row]
    cred_set = filtered_arr[:, [0, i + 1, i + L_snps + 1, i + 2 * L_snps + 1]]
    win = Window.rowsBetween(
        Window.unboundedPreceding, Window.unboundedFollowing
    )
    cred_set = (
        session.spark.createDataFrame(
            cred_set.tolist(),
            ["variantId", "posteriorProbability", "logBF", "beta"],
        )
        .join(
            _join.select(
                "variantId",
                "chromosome",
                "position",
            ),
            "variantId",
        )
        .sort(f.desc("posteriorProbability"))
        .withColumn(
            "locus",
            f.collect_list(
                f.struct(
                    f.col("variantId").cast("string").alias("variantId"),
                    f.col("posteriorProbability")
                    .cast("double")
                    .alias("posteriorProbability"),
                    f.col("logBF").cast("double").alias("logBF"),
                    f.col("beta").cast("double").alias("beta"),
                )
            ).over(win),
        )
        .limit(1)
        .withColumns(
            {
                "studyId": f.lit(_studyId),
                "region": f.lit(_region),
                "credibleSetIndex": f.lit(i + 1),
                "credibleSetlog10BF": f.lit(value * 0.4342944819),
                "finemappingMethod": f.lit("SuSiE-inf"),
            }
        )
        .withColumn(
            "studyLocusId",
            StudyLocus.assign_study_locus_id(
                f.col("studyId"), f.col("variantId")
            ),
        )
        .select(
            "studyLocusId",
            "studyId",
            "region",
            "credibleSetIndex",
            "locus",
            "variantId",
            "chromosome",
            "position",
            "finemappingMethod",
            "credibleSetlog10BF",
        )
    )
    if cred_sets is None:
        cred_sets = cred_set
    else:
        cred_sets = cred_sets.union(cred_set)

In [125]:
order_creds = list(enumerate(susie_output["lbf"]))
order_creds.sort(key=lambda x: x[1], reverse=True)


In [128]:
order_creds

[(0, 731.1532533381637),
 (1, 351.5114266058942),
 (3, 126.85438855538354),
 (2, 51.699811264225424),
 (4, 37.93192592827882),
 (5, 31.009771681812502),
 (6, 23.493131948721746),
 (8, 16.921932293521717),
 (9, 9.320379511652039),
 (7, 1.53384939408551)]

In [133]:
order_creds = list(enumerate(susie_output["lbf"]))
order_creds.sort(key=lambda x: x[1], reverse=True)
order_creds = [
    (item[0], item[1], index + 1) for index, item in enumerate(order_creds)
]
cred_sets = None
counter = 0

In [134]:
order_creds

[(0, 731.1532533381637, 1),
 (1, 351.5114266058942, 2),
 (3, 126.85438855538354, 3),
 (2, 51.699811264225424, 4),
 (4, 37.93192592827882, 5),
 (5, 31.009771681812502, 6),
 (6, 23.493131948721746, 7),
 (8, 16.921932293521717, 8),
 (9, 9.320379511652039, 9),
 (7, 1.53384939408551, 10)]

In [135]:
order_creds = list(enumerate(susie_output["lbf"]))
order_creds.sort(key=lambda x: x[1], reverse=True)
order_creds = [
    (item[0], item[1], index + 1) for index, item in enumerate(order_creds)
]
cred_sets = None
counter = 0
for i, value, cs_index in order_creds:
    if counter > 0 and value < cs_lbf_thr:
        continue
    counter += 1
    sorted_arr = susie_result[
        susie_result[:, i + 1].astype(float).argsort()[::-1]
    ]
    cumsum_arr = np.cumsum(sorted_arr[:, i + 1].astype(float))
    filter_row = np.argmax(cumsum_arr >= 0.99)
    if filter_row == 0 and cumsum_arr[0] < 0.99:
        filter_row = len(cumsum_arr)
    filter_row += 1
    filtered_arr = sorted_arr[:filter_row]
    cred_set = filtered_arr[:, [0, i + 1, i + L_snps + 1, i + 2 * L_snps + 1]]
    win = Window.rowsBetween(
        Window.unboundedPreceding, Window.unboundedFollowing
    )
    cred_set = (
        session.spark.createDataFrame(
            cred_set.tolist(),
            ["variantId", "posteriorProbability", "logBF", "beta"],
        )
        .join(
            _join.select(
                "variantId",
                "chromosome",
                "position",
            ),
            "variantId",
        )
        .sort(f.desc("posteriorProbability"))
        .withColumn(
            "locus",
            f.collect_list(
                f.struct(
                    f.col("variantId").cast("string").alias("variantId"),
                    f.col("posteriorProbability")
                    .cast("double")
                    .alias("posteriorProbability"),
                    f.col("logBF").cast("double").alias("logBF"),
                    f.col("beta").cast("double").alias("beta"),
                )
            ).over(win),
        )
        .limit(1)
        .withColumns(
            {
                "studyId": f.lit(_studyId),
                "region": f.lit(_region),
                "credibleSetIndex": f.lit(cs_index),
                "credibleSetlog10BF": f.lit(value * 0.4342944819),
                "finemappingMethod": f.lit("SuSiE-inf"),
            }
        )
        .withColumn(
            "studyLocusId",
            StudyLocus.assign_study_locus_id(
                f.col("studyId"), f.col("variantId")
            ),
        )
        .select(
            "studyLocusId",
            "studyId",
            "region",
            "credibleSetIndex",
            "locus",
            "variantId",
            "chromosome",
            "position",
            "finemappingMethod",
            "credibleSetlog10BF",
        )
    )
    if cred_sets is None:
        cred_sets = cred_set
    else:
        cred_sets = cred_sets.union(cred_set)

In [139]:
cred_sets.toPandas()

24/03/21 12:47:19 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
24/03/21 12:47:19 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
24/03/21 12:47:19 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
24/03/21 12:47:19 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
24/03/21 12:47:19 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
24/03/21 12:47:19 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
24/03/21 1

,studyLocusId,studyId,region,credibleSetIndex,locus,variantId,chromosome,position,finemappingMethod,credibleSetlog10BF
0,-2446426966894321332,lol,lol,1,"[(18_223185_G_A, 1.0, 734.1977757758872, -0.81...",18_223185_G_A,18,223185,SuSiE-inf,317.535823
1,-6583502620436678966,lol,lol,2,"[(18_225051_C_T, 1.0, 354.5559490436176, 0.550...",18_225051_C_T,18,225051,SuSiE-inf,152.659473
2,-9105124956544629275,lol,lol,3,"[(18_247834_T_G, 1.0, 129.89891099310697, 0.32...",18_247834_T_G,18,247834,SuSiE-inf,55.092161
3,-2446426966894321332,lol,lol,4,"[(18_223185_G_A, 1.0, 54.74433370194885, -0.22...",18_223185_G_A,18,223185,SuSiE-inf,22.452943
4,4044377899302397793,lol,lol,5,"[(18_225964_A_G, 1.0, 40.976448366002245, 0.17...",18_225964_A_G,18,225964,SuSiE-inf,16.473626
5,-2446426966894321332,lol,lol,6,"[(18_223185_G_A, 0.9999999999999858, 34.054294...",18_223185_G_A,18,223185,SuSiE-inf,13.467373
6,-6583502620436678966,lol,lol,7,"[(18_225051_C_T, 0.9999999999674181, 26.537654...",18_225051_C_T,18,225051,SuSiE-inf,10.202938
7,-5393890046996578659,lol,lol,8,"[(18_437232_T_C, 0.9999999789207801, 19.966454...",18_437232_T_C,18,437232,SuSiE-inf,7.349102
8,-5174944260892678388,lol,lol,9,"[(18_351114_C_A, 0.9999295206456712, 12.364831...",18_351114_C_A,18,351114,SuSiE-inf,4.047789


In [141]:
StudyLocus(
            _df=cred_sets,
            _schema=StudyLocus.get_schema(),
        )


StudyLocus(_df=DataFrame[studyLocusId: bigint, studyId: string, region: string, credibleSetIndex: int, locus: array<struct<variantId:string,posteriorProbability:double,logBF:double,beta:double>>, variantId: string, chromosome: string, position: int, finemappingMethod: string, credibleSetlog10BF: double], _schema=StructType([StructField('studyLocusId', LongType(), False), StructField('variantId', StringType(), False), StructField('chromosome', StringType(), True), StructField('position', IntegerType(), True), StructField('region', StringType(), True), StructField('studyId', StringType(), False), StructField('beta', DoubleType(), True), StructField('pValueMantissa', FloatType(), True), StructField('pValueExponent', IntegerType(), True), StructField('effectAlleleFrequencyFromSource', FloatType(), True), StructField('standardError', DoubleType(), True), StructField('subStudyDescription', StringType(), True), StructField('qualityControls', ArrayType(StringType(), False), True), StructField(

In [161]:
def susie_inf_to_studylocus(
    susie_output: dict[str, Any],
    session: Session,
    _studyId: str,
    _region: str,
    _join: DataFrame,
    cs_lbf_thr: float = 2,
) -> StudyLocus:
    """Convert SuSiE-inf output to studyLocus DataFrame.

    Args:
        susie_output (dict[str, Any]): SuSiE-inf output dictionary
        session (Session): Spark session
        _studyId (str): study ID
        _region (str): region
        _join (DataFrame): DataFrame with variant information
        cs_lbf_thr (float): credible set logBF threshold, default is 2

    Returns:
        StudyLocus: StudyLocus object with fine-mapped credible sets
    """
    variants = np.array(
        [row["variantId"] for row in _join.select("variantId").collect()]
    ).reshape(-1, 1)
    PIPs = susie_output["PIP"]
    lbfs = susie_output["lbf_variable"]
    mu = susie_output["mu"]
    susie_result = np.hstack((variants, PIPs, lbfs, mu))

    L_snps = PIPs.shape[1]

    # Extracting credible sets
    order_creds = list(enumerate(susie_output["lbf"]))
    order_creds.sort(key=lambda x: x[1], reverse=True)
    cred_sets = None
    counter = 0
    for i, value in order_creds:
        if counter > 0 and value < cs_lbf_thr:
            counter += 1
            continue
        counter += 1
        sorted_arr = susie_result[
            susie_result[:, i + 1].astype(float).argsort()[::-1]
        ]
        cumsum_arr = np.cumsum(sorted_arr[:, i + 1].astype(float))
        filter_row = np.argmax(cumsum_arr >= 0.99)
        if filter_row == 0 and cumsum_arr[0] < 0.99:
            filter_row = len(cumsum_arr)
        filter_row += 1
        filtered_arr = sorted_arr[:filter_row]
        cred_set = filtered_arr[:, [0, i + 1, i + L_snps + 1, i + 2 * L_snps + 1]]
        win = Window.rowsBetween(
            Window.unboundedPreceding, Window.unboundedFollowing
        )
        cred_set = (
            session.spark.createDataFrame(
                cred_set.tolist(),
                ["variantId", "posteriorProbability", "logBF", "beta"],
            )
            .join(
                _join.select(
                    "variantId",
                    "chromosome",
                    "position",
                ),
                "variantId",
            )
            .sort(f.desc("posteriorProbability"))
            .withColumn(
                "locus",
                f.collect_list(
                    f.struct(
                        f.col("variantId").cast("string").alias("variantId"),
                        f.col("posteriorProbability")
                        .cast("double")
                        .alias("posteriorProbability"),
                        f.col("logBF").cast("double").alias("logBF"),
                        f.col("beta").cast("double").alias("beta"),
                    )
                ).over(win),
            )
            .limit(1)
            .withColumns(
                {
                    "studyId": f.lit(_studyId),
                    "region": f.lit(_region),
                    "credibleSetIndex": f.lit(counter),
                    "credibleSetlog10BF": f.lit(value * 0.4342944819),
                    "finemappingMethod": f.lit("SuSiE-inf"),
                }
            )
            .withColumn(
                "studyLocusId",
                StudyLocus.assign_study_locus_id(
                    f.col("studyId"), f.col("variantId")
                ),
            )
            .select(
                "studyLocusId",
                "studyId",
                "region",
                "credibleSetIndex",
                "locus",
                "variantId",
                "chromosome",
                "position",
                "finemappingMethod",
                "credibleSetlog10BF",
            )
        )
        if cred_sets is None:
            cred_sets = cred_set
        else:
            cred_sets = cred_sets.union(cred_set)
    return StudyLocus(
        _df=cred_sets,
        _schema=StudyLocus.get_schema(),
    )


In [167]:
L1=susie_inf_to_studylocus(susie_output=susie_output, session=session, _studyId="lol", _region="lol1", _join=gwas_df, cs_lbf_thr=2)

In [168]:
L1.df.toPandas()["locus"][0]

24/03/21 13:06:21 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
24/03/21 13:06:21 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
24/03/21 13:06:21 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
24/03/21 13:06:21 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
24/03/21 13:06:21 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
24/03/21 13:06:21 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
24/03/21 1

[Row(variantId='18_247834_T_G', posteriorProbability=1.0, logBF=1519.2816760403173, beta=0.16981972321672212)]

In [169]:
L1.df.toPandas()

,studyLocusId,studyId,region,credibleSetIndex,locus,variantId,chromosome,position,finemappingMethod,credibleSetlog10BF
0,-9105124956544629275,lol,lol1,1,"[(18_247834_T_G, 1.0, 1519.2816760403173, 0.16...",18_247834_T_G,18,247834,SuSiE-inf,658.493429
1,-6583502620436678966,lol,lol1,2,"[(18_225051_C_T, 1.0, 1385.2122979160038, 0.16...",18_225051_C_T,18,225051,SuSiE-inf,600.267838
2,-2446426966894321332,lol,lol1,3,"[(18_223185_G_A, 1.0, 1353.2738672223911, -0.1...",18_223185_G_A,18,223185,SuSiE-inf,586.397154
3,-2446426966894321332,lol,lol1,4,"[(18_223185_G_A, 1.0, 1293.5321746817358, -0.1...",18_223185_G_A,18,223185,SuSiE-inf,560.451666
4,-2446426966894321332,lol,lol1,5,"[(18_223185_G_A, 1.0, 1287.311459187472, -0.15...",18_223185_G_A,18,223185,SuSiE-inf,557.750044
5,-6583502620436678966,lol,lol1,6,"[(18_225051_C_T, 1.0, 1134.9666372097058, 0.14...",18_225051_C_T,18,225051,SuSiE-inf,491.587528
6,-2446426966894321332,lol,lol1,7,"[(18_223185_G_A, 1.0, 1053.8611124290842, -0.1...",18_223185_G_A,18,223185,SuSiE-inf,456.363847
7,-6583502620436678966,lol,lol1,8,"[(18_225051_C_T, 1.0, 910.6953159870186, 0.131...",18_225051_C_T,18,225051,SuSiE-inf,394.187731
8,4044377899302397793,lol,lol1,9,"[(18_225964_A_G, 1.0, 846.9966614292723, 0.126...",18_225964_A_G,18,225964,SuSiE-inf,366.523757
9,-2446426966894321332,lol,lol1,10,"[(18_223185_G_A, 1.0, 615.4329074525784, -0.10...",18_223185_G_A,18,223185,SuSiE-inf,265.956896
